In [36]:
import sys
from pathlib import Path

# Find repo root (directory that contains src/ and requirements.txt)
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

# Returns True if .env file was found; False if not (Docker may already inject vars via env_file)
loaded = load_dotenv(ROOT / ".env")
print(f"Project root: {ROOT}")
print(f"load_dotenv: {loaded}  (.env path: {ROOT / '.env'})")

Project root: /app
load_dotenv: True  (.env path: /app/.env)


In [37]:
from importlib.metadata import version,PackageNotFoundError
from langgraph.graph import StateGraph,START,END

try:
    print(f"LangGraph version: {version('langgraph')}")
except PackageNotFoundError:
    print("LangGraph version: (installed,version metadata available)")
print("Imports OK: StateGraph, START, END")

LangGraph version: 0.2.53
Imports OK: StateGraph, START, END


In [38]:
from typing import TypedDict

#data type data share betwen need
class State(TypedDict):
    message:str


#node/Function: function that read and update state
def greet (state:State) -> State:
    return {
        "message":f"hello,{state['message']}!"
    }

graph = StateGraph(State)#create empty graph builder
graph.add_node("greet",greet)
graph.add_edge(START,"greet")
graph.add_edge("greet",END)

app = graph.compile() #freeze graphs into runnable application
result = app.invoke(
    {
        "message":"qayyum"
    })
result

{'message': 'hello,qayyum!'}

In [69]:
# --- Imports ---
from langchain_core.prompts import ChatPromptTemplate   # reusable prompt template with {variables}
from langchain_core.output_parsers import StrOutputParser  # converts LLM response object → plain string
from src.config import OPENAI_API_KEY                   # reads OPENAI_API_KEY from .env / Docker env
from src.llms.openai_chat import make_openai_chat       # project helper that returns a configured ChatOpenAI
from langchain_core.runnables import RunnableLambda

if OPENAI_API_KEY:
    llm = make_openai_chat() #create openai.
    prompt = ChatPromptTemplate.from_template(
        "Say hello to {name} in short sentence.and say random wisdom"
    )

    lcel_llm_chain = prompt | llm | StrOutputParser()
    lcel_reply = lcel_llm_chain.invoke({
        "name":"Qayyum"
    })
    print("LCEL reply:",lcel_reply)

    class LlmState(TypedDict):
        name:str #input from user
        reply:str #Llm response

    def llm_node(state:LlmState) ->LlmState:
        reply = lcel_llm_chain.invoke({
            "name":state['name']
        })
        return {"reply":reply}

    llm_graph = StateGraph(LlmState)
    llm_graph.add_node("llm",llm_node)
    llm_graph.add_edge(START,"llm")
    llm_graph.add_edge("llm",END)

    llm_app = llm_graph.compile()
    graph_reply = llm_app.invoke({
        "name":"Qayyum",
        "reply":""
    })
    print("Graph + LLM:", graph_reply["reply"])
print("Skipping LLM demo — set OPENAI_API_KEY in .env to run this cell.")


LCEL reply: Hello, Qayyum! Remember, the journey of a thousand miles begins with a single step.
Graph + LLM: Hello, Qayyum! Remember, the journey of a thousand miles begins with a single step.
Skipping LLM demo — set OPENAI_API_KEY in .env to run this cell.


In [70]:
#Scenario A : Support Ticker router (no LLM,no LCEL)
# Demonstrates conditional routing — the main reason to use LangGraph over a single chain
#START → classify → [billing | technical | general] → END

class TicketState(TypedDict):
    message:str #user ticket
    category:str #fill by classify node
    response:str #fill by the specialist node

def classify_node(state:TicketState) -> TicketState:
    # Node 1: plain Python logic — no LCEL needed
    msg = state['message'].lower()
    if "bill" in msg or "payment" in msg:
        category = "billing"
    if "error" in msg or "bug" in msg:
        category = "general"
    return {"category":category}

def billing_node(state:TicketState) -> TicketState:
    return {
        "response": "Billing: we'll review your invoice within 24 hours."
    }
    
def technical_node(state:TicketState) -> TicketState:
    return {
        "response": "Tech Support: please share error logs and step to produce."
    }

def general_node(state:TicketState) -> TicketState:
    return {
        "response": "Support: thanks for reaching out - an agent will follow up shortly"
    }

def route_ticket(state:TicketState) -> str:
    return state["category"]


ticket_graph = StateGraph(TicketState)

#setup node
ticket_graph.add_node("classify",classify_node)
ticket_graph.add_node("billing",billing_node)
ticket_graph.add_node("technical", technical_node)
ticket_graph.add_node("general",general_node)

ticket_graph.add_edge(START,"classify")
ticket_graph.add_conditional_edges("classify",route_ticket)
ticket_graph.add_edge("billing",END)
ticket_graph.add_edge("technical",END)
ticket_graph.add_edge("general",END)

ticket_app = ticket_graph.compile()

for msg in ["I have a billing issue with my invoice","the app throws an error on login"]:
    out = ticket_app.invoke({
        "message":msg,
        "category":"",
        "response":""
    })
    print(f"input:{msg}")
    print(f"Route:{out['category']} -> {out['response']}\n")







input:I have a billing issue with my invoice
Route:billing -> Billing: we'll review your invoice within 24 hours.

input:the app throws an error on login
Route:general -> Support: thanks for reaching out - an agent will follow up shortly



In [85]:
# --- Scenario B: validate → generate (LCEL) → format ---
# Shows LCEL inside ONE node of a multi-node graph (no API key — RunnableLambda stand-in for llm chain)
#START → validate → generate (LCEL) → format → END

class PipelineState(TypedDict):
    name:str
    valid:bool
    draft:str
    final:str

def validate_node(state:PipelineState)->PipelineState:
    return {
        "valid":len(state["name"].strip()) > 0
    }

def route_after_validate(state:PipelineState) ->str:
    if state["valid"]:
        return "generate"
    else:
        return END

# Step 2: LCEL chain built FIRST, then called inside the node (same pattern as Part 4)
generate_chain = RunnableLambda(
    lambda x: f"Hello,{x['name']}! welcome to Langraph!"
)

def generate_node(state:PipelineState) -> PipelineState:
    # Node wraps the LCEL chain — graph registers this function, not the chain itself

    draft = generate_chain.invoke({
        "name":state["name"]
    })
    return {
        "draft":draft
    }

def format_node(state:PipelineState) -> PipelineState:
    # Step 3: plain Python post-processing
    return {
        "final":state['draft'].upper()
    }

#Declara graph and nodes
pipeline_graph = StateGraph(PipelineState)
pipeline_graph.add_node("validate",validate_node)
pipeline_graph.add_node("generate",generate_node)
pipeline_graph.add_node("format",format_node)

pipeline_graph.add_edge(START,"validate")
pipeline_graph.add_conditional_edges("validate",route_after_validate)
pipeline_graph.add_edge("generate","format")
pipeline_graph.add_edge("format",END)

pipeline_app = pipeline_graph.compile()

# Valid input: runs all three nodes
good = pipeline_app.invoke({
    "name":"Abdul Qayyum",
    "valid":False,
    "draft":"",
    "final":""
})
print("valid input:",good)

# Invalid input: validate → END (generate and format are skipped)
bad = pipeline_app.invoke({
    "name":"",
    "valid":False,
    "draft":"",
    "final":""
})
print("invalid input:",bad)


valid input: {'name': 'Abdul Qayyum', 'valid': True, 'draft': 'Hello,Abdul Qayyum! welcome to Langraph!', 'final': 'HELLO,ABDUL QAYYUM! WELCOME TO LANGRAPH!'}
invalid input: {'name': '', 'valid': False, 'draft': '', 'final': ''}
